# 모델 선택 기준

정확도, 속도, 메모리, 라이선스 조건이 다를 때 어떻게 모델 후보를 비교할까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 비교표 만들기

아래 값은 본 실습에서 사용한 모델 카드의 공개 정보와 현재 환경의 파라미터 수입니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
from transformers import AutoModelForCausalLM, AutoTokenizer

candidates = [
    {"id": "Qwen/Qwen2.5-0.5B-Instruct", "task": "text-generation", "license": "apache-2.0"},
    {"id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "task": "sentence-similarity", "license": "apache-2.0"},
    {"id": "klue/bert-base", "task": "fill-mask", "license": "CC-BY-SA-4.0"},
]
for item in candidates:
    print(item)

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part1/ch05-llm-principles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'task': 'text-generation', 'license': 'apache-2.0'}
{'id': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'task': 'sentence-similarity', 'license': 'apache-2.0'}
{'id': 'klue/bert-base', 'task': 'fill-mask', 'license': 'CC-BY-SA-4.0'}


## 현재 환경에서 실행 가능성 확인

생성 모델을 로드하여 파라미터 수와 토큰화 시간을 측정합니다. 이 시간은 현재 컴퓨터의 캐시 상태에만 해당합니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import time
model_id = candidates[0]["id"]
revision = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
start = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision)
load_seconds = time.perf_counter() - start
text = "한국어 문장을 토큰화합니다. " * 100
start = time.perf_counter()
ids = tokenizer.encode(text)
tokenize_ms = (time.perf_counter() - start) * 1000
print("parameters:", f"{model.num_parameters():,}")
print("cached load seconds:", round(load_seconds, 3))
print("tokenize:", len(ids), "tokens in", round(tokenize_ms, 3), "ms")

parameters: 494,032,768
cached load seconds: 0.382
tokenize: 1202 tokens in 0.919 ms


## 필수 조건으로 후보 거르기

생성 작업과 Apache-2.0 라이선스를 필수 조건으로 두면 조건을 충족하는 후보만 남습니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
required_task = "text-generation"
required_license = "apache-2.0"
eligible = [item for item in candidates if item["task"] == required_task and item["license"] == required_license]
print("eligible:", eligible)
print("선택 전에는 자체 질의셋으로 품질과 속도를 추가 측정해야 합니다.")

eligible: [{'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'task': 'text-generation', 'license': 'apache-2.0'}]
선택 전에는 자체 질의셋으로 품질과 속도를 추가 측정해야 합니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [4]:
assert len(candidates) == 3 and len(eligible) == 1
assert model.num_parameters() == 494_032_768
print("검증: 필수 조건으로 3개 후보 중 1개를 걸렀습니다.")

검증: 필수 조건으로 3개 후보 중 1개를 걸렀습니다.
